In [52]:
import gensim.downloader as api
glove = api.load("glove-wiki-gigaword-50")   # downloads once, then loads from disk

import numpy as np
import pandas as pd

from nltk.tokenize import word_tokenize

In [53]:
data = pd.read_csv('Day1_Dataset.csv')
data.head()

,review_id,review,sentiment
0,1,"<br />A really painful movie, I am not recom...",negative
1,2,"@cinema_guy The story was dull, the jokes were...",negative
2,3,The leading actors played their roles perfec...,positive
3,4,THE PLOT WAS CONFUSING AND THE PERFORMANCES ...,negative
4,5,"I did not enjoy a single minute, the charact...",negative


In [54]:
def review_to_vector(text):
    words = word_tokenize(text.lower())
    vectors = [glove[w] for w in words if w in glove]   # skip words GloVe doesn't know
    return np.mean(vectors, axis=0)

In [55]:
review_to_vector(data["review"][0])

array([-0.02880576,  0.42133304,  0.35741442, -0.07016934,  0.05419566,
        0.06088369,  0.1250345 , -0.4816865 , -0.4800643 , -0.10290212,
       -0.03803034,  0.33432114, -0.230982  , -0.32005647,  0.4859814 ,
        0.18549411, -0.07544956,  0.232877  , -0.36549583, -0.10722402,
       -0.20553862, -0.09366177,  0.5142576 ,  0.34612235,  0.12156852,
       -0.9290143 , -0.45370886, -0.033432  ,  0.36262426, -0.14213656,
        2.3738334 ,  0.38194752, -0.14495918,  0.24195732,  0.02987138,
       -0.19180924,  0.37832493, -0.26361805,  0.29537967, -0.10566323,
        0.24106662,  0.02902444,  0.15294865,  0.16282728, -0.14274588,
        0.07623511,  0.08597181, -0.00305699,  0.32985935,  0.5719373 ],
      dtype=float32)

In [56]:
# step4: Data Preprocessing

import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Create stopwords set
stop_words = set(stopwords.words('english'))

# Create the lemmatizer object once
lemmatizer = WordNetLemmatizer()

# Keep negation words because they affect sentiment
stop_words = stop_words - {"not", "no", "nor"}

def preprocess(text):
    text = text.lower()
    text = re.sub(r"<.*?>|http\S+|www\S+|@\w+|[^a-z\s]", "", text)
    words = word_tokenize(text)
    words = [
        word for word in words
        if word not in stopwords.words('english')
    ]
    words = [lemmatizer.lemmatize(lemmatizer.lemmatize(word, pos='v'), pos='n') for word in words]
    words = [word for word in words if len(word)>1]
    return " ".join(words)

In [57]:
# Apply preprocessing
data['cleaned'] = data['review'].apply(preprocess)

data.head()

,review_id,review,sentiment,cleaned
0,1,"<br />A really painful movie, I am not recom...",negative,really painful movie recommend anyone worst ti...
1,2,"@cinema_guy The story was dull, the jokes were...",negative,story dull joke funny end bad
2,3,The leading actors played their roles perfec...,positive,lead actor play role perfectly delightful surp...
3,4,THE PLOT WAS CONFUSING AND THE PERFORMANCES ...,negative,plot confuse performance weak story dull joke ...
4,5,"I did not enjoy a single minute, the charact...",negative,enjoy single minute character felt fake movie ...


In [58]:
data['vectors'] = data['review'].apply(review_to_vector)

In [59]:
data.head()

,review_id,review,sentiment,cleaned,vectors
0,1,"<br />A really painful movie, I am not recom...",negative,really painful movie recommend anyone worst ti...,"[-0.028806, 0.421333, 0.357414, -0.070169, 0.0..."
1,2,"@cinema_guy The story was dull, the jokes were...",negative,story dull joke funny end bad,"[0.048955, 0.043931, -0.267331, -0.098886, 0.2..."
2,3,The leading actors played their roles perfec...,positive,lead actor play role perfectly delightful surp...,"[0.211352, 0.11028, -0.210544, -0.158046, 0.49..."
3,4,THE PLOT WAS CONFUSING AND THE PERFORMANCES ...,negative,plot confuse performance weak story dull joke ...,"[0.259968, 0.032844, -0.19309, -0.075506, 0.29..."
4,5,"I did not enjoy a single minute, the charact...",negative,enjoy single minute character felt fake movie ...,"[0.207716, 0.124631, -0.005756, -0.09657, 0.46..."


In [60]:
X = data['vectors']

y = data['cleaned']

In [61]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X, y
)

print("Training data:", len(X_train))
print("Testing data:", len(X_test))

Training data: 315
Testing data: 105


In [62]:
X = np.vstack(X.values)
X

array([[-0.02880576,  0.42133304,  0.35741442, ..., -0.00305699,
         0.32985935,  0.5719373 ],
       [ 0.048955  ,  0.04393098, -0.26733115, ..., -0.2323581 ,
         0.06582375,  0.05102793],
       [ 0.21135215,  0.11027984, -0.21054381, ..., -0.12539393,
        -0.12460197,  0.2554479 ],
       ...,
       [ 0.11421781,  0.10408745,  0.03263698, ..., -0.40927848,
         0.03821694,  0.27055627],
       [ 0.06877924,  0.29406753, -0.03840259, ..., -0.28545797,
        -0.00813698,  0.17306453],
       [ 0.24011308,  0.07556082, -0.29009452, ..., -0.15622036,
        -0.02644185,  0.11825847]], shape=(420, 50), dtype=float32)

In [63]:
# step7: model training
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression()
logistic_model.fit(X_train, y_train)

ValueError: setting an array element with a sequence.

In [ ]:
print("X_train type:", type(X_train))
print("X_train shape:", X_train.shape)

print("y_train type:", type(y_train))
print("y_train shape:", y_train.shape)

print("\nX_train sample:")
print(X_train[:5])

print("\ny_train sample:")
print(y_train[:5])